<a href="https://colab.research.google.com/github/lgonperez3-glitch/MLSM-labs/blob/main/Lab3/lab3_autoencoder_mtat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — Autoencoder convolucional para MagnaTagATune

**Machine Learning for Sound and Music — UPF**

Este notebook adapta el autoencoder convolucional de la sesión 06_01 al conjunto de datos MagnaTagATune (MTAT). El objetivo es aprender a reconstruir segmentos de audio sin utilizar etiquetas: la propia señal de entrada actúa como objetivo de reconstrucción.

## Qué vamos a hacer

1. Preparar y cargar los audios de MTAT.
2. Trabajar con recortes de un segundo para controlar la longitud de entrada y el coste computacional.
3. Entrenar un autoencoder de referencia con pérdida MSE.
4. Probar una arquitectura más profunda y una pérdida espectral multiescala, manteniendo intacta la compresión del bottleneck.
5. Comparar los modelos mediante el MSE del conjunto de validación y escuchar/visualizar reconstrucciones.

**Importante:** las celdas de entrenamiento deben ejecutarse para obtener resultados reales. No se incluyen métricas inventadas ni resultados simulados.

## 1. Diseño experimental y decisiones

### Por qué utilizamos recortes de un segundo

El notebook de clase trabaja con señales mono a 16 kHz y una longitud de 16 000 muestras. MTAT contiene clips considerablemente más largos. En lugar de introducir señales de longitud variable en la red, extraeremos un segmento de un segundo de cada clip en cada acceso al conjunto de entrenamiento.

- Frecuencia de muestreo: 16 000 Hz.
- Canales: mono.
- Longitud de entrada: 16 000 muestras (1 segundo).
- Recorte de entrenamiento: posición aleatoria para que el modelo no vea siempre el mismo segundo.
- Recorte de validación: posición central fija para que la comparación sea reproducible.

Esta decisión conserva la longitud de entrada del ejemplo de clase y, por tanto, permite mantener el tamaño del bottleneck original.

### Restricción del bottleneck

Se conserva el valor de referencia:

- Entrada: 16 000 muestras.
- Bottleneck: 512 valores.
- Tasa de compresión según la definición del notebook de clase: 16 000 / 512 = **31,25×**.

No modificaremos estos dos valores en ninguno de los experimentos.

### Comparación

| Experimento | Arquitectura | Pérdida usada para entrenar |
|---|---|---|
| A — Baseline | 3 bloques convolucionales | MSE |
| B — Mejorado | Más convoluciones por bloque, mismo número de reducciones | MSE + pérdida espectral multiescala |

La pérdida espectral ayuda a comparar el contenido en frecuencia de la señal original y la reconstruida. Aun así, la selección del mejor modelo y la comparación final se harán mediante **MSE de validación**, tal como exige el enunciado.

## 2. Instalación de dependencias

El entorno recomendado es Google Colab con GPU. PyTorch suele venir preinstalado en Colab; por eso no forzamos aquí una reinstalación que pudiera cambiar la versión de CUDA. Instalamos las librerías auxiliares y FFmpeg para poder leer los archivos MP3.

In [ ]:
# Instalamos FFmpeg, necesario para decodificar los audios MP3 de MTAT.
!apt-get -qq update
!apt-get -qq install -y ffmpeg

# Instalamos las librerías de procesamiento de audio, datos y visualización.
!pip -q install librosa==0.10.2.post1 scikit-learn matplotlib pandas tqdm

## 3. Importaciones y configuración

Fijamos una semilla para que la partición entrenamiento/validación y la inicialización de los modelos sean reproducibles en la medida de lo posible. Las operaciones de GPU pueden introducir pequeñas diferencias entre ejecuciones.

In [ ]:
import os
import random
import shutil
import subprocess
from pathlib import Path
from google.colab import drive

# Montamos Google Drive y creamos la carpeta de resultados.
drive.mount('/content/drive')
RESULTS_DIR = Path('/content/drive/MyDrive/MLSM/Lab 3/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Los datos originales se mantienen en el almacenamiento temporal de Colab.
DATA_ROOT = Path('/content/mtat')
DOWNLOAD_DIR = DATA_ROOT / 'downloads'
AUDIO_DIR = DATA_ROOT / 'audio'
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
PARTS = ['mp3.zip.001', 'mp3.zip.002', 'mp3.zip.003']

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from IPython.display import Audio, display, Markdown

# Semilla común para todos los experimentos.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Seleccionamos GPU si está disponible; si no, el notebook puede ejecutarse en CPU,
# aunque el entrenamiento será considerablemente más lento.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Parámetros de audio y del experimento.
SAMPLE_RATE = 16_000
SEGMENT_SECONDS = 1.0
INPUT_LENGTH = int(SAMPLE_RATE * SEGMENT_SECONDS)

BOTTLENECK_SIZE = 512
COMPRESSION_RATE = INPUT_LENGTH / BOTTLENECK_SIZE

BATCH_SIZE = 32
NUM_EPOCHS = 8
LEARNING_RATE = 1e-3
PATIENCE = 3
NUM_WORKERS = 2

print(f"Longitud de entrada: {INPUT_LENGTH} muestras")
print(f"Bottleneck: {BOTTLENECK_SIZE} valores")
print(f"Tasa de compresión: {COMPRESSION_RATE:.2f}x")

## 4. Descarga y preparación de MagnaTagATune

El servidor oficial de MIRG distribuye el audio en tres volúmenes: mp3.zip.001, mp3.zip.002 y mp3.zip.003. Los descargamos, los concatenamos en el orden correcto y extraemos el archivo resultante.

La descarga ocupa varios GB y puede tardar. Si se interrumpe, el parámetro de continuación de wget permite reanudar cada parte. El almacenamiento temporal de Colab se pierde al terminar la sesión; si no quieres volver a descargar los datos, puedes cambiar DATA_ROOT a una carpeta de Google Drive después de montarla.

Para esta tarea de reconstrucción no necesitamos las etiquetas musicales: el objetivo de cada ejemplo es el propio segmento de audio. Por eso basta con preparar los archivos de audio.

In [ ]:
# Comprobamos que las rutas existen incluso si esta celda se ejecuta por separado.
if "DOWNLOAD_DIR" not in globals() or "PARTS" not in globals():
    from pathlib import Path
    import subprocess
    DATA_ROOT = Path('/content/mtat')
    DOWNLOAD_DIR = DATA_ROOT / 'downloads'
    AUDIO_DIR = DATA_ROOT / 'audio'
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    AUDIO_DIR.mkdir(parents=True, exist_ok=True)
    PARTS = ['mp3.zip.001', 'mp3.zip.002', 'mp3.zip.003']

# Descargamos las tres partes oficiales. wget -c reanuda archivos parciales.
BASE_URL = "https://mirg.city.ac.uk/datasets/magnatagatune"
for part in PARTS:
    subprocess.run(
        ["wget", "-c", f"{BASE_URL}/{part}", "-P", str(DOWNLOAD_DIR)],
        check=True
    )
    destination = DOWNLOAD_DIR / part
    print(f"{part}: {destination.stat().st_size / (1024 ** 3):.2f} GB")


In [ ]:
# Nos aseguramos de que las rutas estén definidas antes de concatenar los ZIP.
if "DOWNLOAD_DIR" not in globals() or "AUDIO_DIR" not in globals() or "PARTS" not in globals():
    from pathlib import Path
    DATA_ROOT = Path('/content/mtat')
    DOWNLOAD_DIR = DATA_ROOT / 'downloads'
    AUDIO_DIR = DATA_ROOT / 'audio'
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    AUDIO_DIR.mkdir(parents=True, exist_ok=True)
    PARTS = ['mp3.zip.001', 'mp3.zip.002', 'mp3.zip.003']

# Si falta alguna parte, mostramos una instrucción clara en vez de un NameError.
missing_parts = [part for part in PARTS if not (DOWNLOAD_DIR / part).is_file()]
if missing_parts:
    raise FileNotFoundError(
        f"Faltan estas partes del dataset: {missing_parts}. "
        "Ejecuta primero la celda anterior de descarga."
    )

# Unimos las tres partes para obtener el ZIP completo.
MERGED_ARCHIVE = DOWNLOAD_DIR / "mp3.zip"
with open(MERGED_ARCHIVE, "wb") as output_file:
    for part in PARTS:
        with open(DOWNLOAD_DIR / part, "rb") as input_file:
            shutil.copyfileobj(input_file, output_file)

# Verificamos la integridad del ZIP antes de extraerlo.
subprocess.run(["unzip", "-tq", str(MERGED_ARCHIVE)], check=True)

# Extraemos siempre para completar posibles extracciones interrumpidas.
subprocess.run(
    ["unzip", "-q", "-o", str(MERGED_ARCHIVE), "-d", str(AUDIO_DIR)],
    check=True
)

print("Carpeta de audio:", AUDIO_DIR)


In [ ]:
# Localizamos todos los MP3, independientemente de cómo estén organizadas
# las subcarpetas dentro del archivo comprimido.
audio_paths = sorted(AUDIO_DIR.rglob("*.mp3"))

if len(audio_paths) == 0:
    raise RuntimeError(
        "No se han encontrado archivos MP3. Comprueba la descarga y la extracción "
        "del archivo oficial de MTAT."
    )

print(f"Archivos MP3 encontrados: {len(audio_paths):,}")
print("Ejemplo:", audio_paths[0])

# Mostramos el tamaño aproximado de la carpeta para conocer el espacio utilizado.
total_audio_gb = sum(path.stat().st_size for path in audio_paths) / (1024 ** 3)
print(f"Tamaño total de los MP3 encontrados: {total_audio_gb:.2f} GB")

## 5. Separación de entrenamiento y validación

Separamos las rutas de audio antes de extraer los recortes. De este modo, un mismo archivo MP3 no puede aparecer a la vez en entrenamiento y validación.

La partición se realiza a nivel de clip y con una semilla fija. No utilizamos las etiquetas de MTAT para entrenar: el autoencoder es un modelo de aprendizaje autosupervisado y aprende a reconstruir la entrada.

In [ ]:
# Volvemos a escanear el directorio justo antes de crear los conjuntos.
# Así evitamos conservar rutas de una extracción anterior o archivos eliminados.
audio_paths = sorted(
    path for path in AUDIO_DIR.rglob("*.mp3")
    if path.is_file() and path.stat().st_size > 0
)

if not audio_paths:
    raise RuntimeError(
        "No hay archivos MP3 disponibles en AUDIO_DIR. Ejecuta de nuevo las celdas "
        "de descarga y extracción antes de continuar."
    )

train_paths, val_paths = train_test_split(
    audio_paths,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

print(f"Archivos MP3 válidos encontrados: {len(audio_paths):,}")
print(f"Clips de entrenamiento: {len(train_paths):,}")
print(f"Clips de validación: {len(val_paths):,}")
print(f"Proporción de validación: {len(val_paths) / len(audio_paths):.1%}")

## 6. Dataset de PyTorch: lectura y recorte de audio

Creamos una clase propia que carga únicamente el segundo que necesitamos, en lugar de mantener todos los audios completos en memoria.

1. Elegimos un desplazamiento aleatorio para entrenamiento o el centro del clip para validación.
2. Leemos un segundo en mono y a 16 kHz.
3. Si el fragmento devuelto es más corto de lo esperado, completamos con ceros.
4. Normalizamos el pico del segmento para mantener una escala estable.

El recorte aleatorio aporta variación entre épocas sin modificar la duración que recibe el autoencoder.

In [ ]:
class MTATAudioDataset(Dataset):
    """Carga segmentos mono de un segundo desde los clips MP3 de MTAT."""

    def __init__(self, paths, training, sample_rate=16_000, segment_seconds=1.0):
        self.paths = list(paths)
        self.training = training
        self.sample_rate = sample_rate
        self.segment_seconds = segment_seconds
        self.segment_length = int(sample_rate * segment_seconds)

        # Los clips estándar de MTAT duran aproximadamente 29 segundos.
        # Dejamos un margen para evitar pedir recortes fuera del final.
        self.max_offset = 27.0

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]

        # En entrenamiento cambiamos la posición del recorte en cada acceso.
        # En validación usamos siempre el mismo punto central.
        if self.training:
            offset = random.uniform(0.0, self.max_offset)
        else:
            offset = self.max_offset / 2.0

        try:
            audio, _ = librosa.load(
                path,
                sr=self.sample_rate,
                mono=True,
                offset=offset,
                duration=self.segment_seconds
            )
        except Exception as error:
            raise RuntimeError(f"No se pudo leer el archivo {path}: {error}")

        # Convertimos a longitud fija: recortamos si sobra audio y rellenamos
        # con silencio si el fragmento es más corto de un segundo.
        audio = np.asarray(audio, dtype=np.float32)
        if len(audio) > self.segment_length:
            audio = audio[:self.segment_length]
        elif len(audio) < self.segment_length:
            audio = np.pad(audio, (0, self.segment_length - len(audio)))

        # Normalización por pico, igual que en el ejemplo TinySOL de clase.
        peak = np.max(np.abs(audio))
        if peak > 1e-8:
            audio = audio / peak

        return torch.from_numpy(audio.copy())


train_dataset = MTATAudioDataset(train_paths, training=True)
val_dataset = MTATAudioDataset(val_paths, training=False)

print("Tamaño del conjunto de entrenamiento:", len(train_dataset))
print("Tamaño del conjunto de validación:", len(val_dataset))

In [ ]:
# DataLoaders: el entrenamiento mezcla los clips; validación conserva el orden.
# pin_memory acelera la transferencia a GPU cuando CUDA está disponible.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=False
)

# Comprobamos las dimensiones antes de crear la red.
example_batch = next(iter(train_loader))
print("Forma de un batch:", example_batch.shape)
assert example_batch.shape[1] == INPUT_LENGTH
display(Audio(example_batch[0].numpy(), rate=SAMPLE_RATE))

## 7. Arquitectura del autoencoder

El modelo sigue la estructura del notebook de clase:

- **Encoder:** tres bloques convolucionales que extraen características y reducen progresivamente la longitud temporal mediante MaxPool1d.
- **Bottleneck:** una capa lineal que comprime la representación a 512 valores.
- **Decoder:** una capa lineal y tres operaciones MaxUnpool1d que recuperan la resolución temporal, seguidas de convoluciones.

Guardamos los índices de MaxPool1d para que MaxUnpool1d pueda colocar las activaciones en las posiciones correspondientes. Es una estrategia de reconstrucción inspirada directamente en la implementación de referencia.

En el modelo mejorado añadimos una convolución adicional en cada bloque del encoder y del decoder. **No añadimos reducciones de resolución ni cambiamos la dimensión 512 del bottleneck.**

In [ ]:
class AudioAutoencoder(nn.Module):
    """Autoencoder 1D con tres reducciones temporales y bottleneck fijo."""

    def __init__(self, input_length=16_000, bottleneck_size=512, channels=8,
                 channel_mult=2, deeper=False):
        super().__init__()

        if input_length % 8 != 0:
            raise ValueError("input_length debe ser divisible por 8.")
        self.input_length = input_length
        self.bottleneck_size = bottleneck_size
        self.deeper = deeper
        kernel_size = 5
        padding = kernel_size // 2

        # ----- Encoder: bloque 1 -----
        self.enc_conv1 = nn.Conv1d(1, channels, kernel_size, padding=padding)
        self.enc_norm1 = nn.BatchNorm1d(channels)
        self.enc_act1 = nn.PReLU()
        self.enc_extra1 = (
            nn.Sequential(nn.Conv1d(channels, channels, kernel_size, padding=padding),
                          nn.BatchNorm1d(channels), nn.PReLU())
            if deeper else nn.Identity()
        )
        self.pool1 = nn.MaxPool1d(2, stride=2, return_indices=True)

        # ----- Encoder: bloque 2 -----
        self.enc_conv2 = nn.Conv1d(channels, channels * channel_mult, kernel_size, padding=padding)
        self.enc_norm2 = nn.BatchNorm1d(channels * channel_mult)
        self.enc_act2 = nn.PReLU()
        self.enc_extra2 = (
            nn.Sequential(nn.Conv1d(channels * channel_mult, channels * channel_mult,
                                    kernel_size, padding=padding),
                          nn.BatchNorm1d(channels * channel_mult), nn.PReLU())
            if deeper else nn.Identity()
        )
        self.pool2 = nn.MaxPool1d(2, stride=2, return_indices=True)

        # ----- Encoder: bloque 3 -----
        self.enc_conv3 = nn.Conv1d(channels * channel_mult,
                                   channels * channel_mult ** 2,
                                   kernel_size, padding=padding)
        self.enc_norm3 = nn.BatchNorm1d(channels * channel_mult ** 2)
        self.enc_act3 = nn.PReLU()
        self.enc_extra3 = (
            nn.Sequential(nn.Conv1d(channels * channel_mult ** 2,
                                    channels * channel_mult ** 2,
                                    kernel_size, padding=padding),
                          nn.BatchNorm1d(channels * channel_mult ** 2), nn.PReLU())
            if deeper else nn.Identity()
        )
        self.pool3 = nn.MaxPool1d(2, stride=2, return_indices=True)

        # Tras tres MaxPool1d(2), la longitud queda dividida entre ocho.
        encoded_length = input_length // 8
        encoded_channels = channels * channel_mult ** 2
        flattened_size = encoded_length * encoded_channels

        # Esta capa conserva el bottleneck solicitado: 512 dimensiones.
        self.enc_fc = nn.Linear(flattened_size, bottleneck_size)
        self.dec_fc = nn.Linear(bottleneck_size, flattened_size)

        # ----- Decoder: operaciones inversas de resolución -----
        self.unpool3 = nn.MaxUnpool1d(2, stride=2)
        self.dec_deconv3 = nn.Conv1d(encoded_channels, channels * channel_mult,
                                     kernel_size, padding=padding)
        self.dec_act3 = nn.PReLU()
        self.dec_extra3 = (
            nn.Sequential(nn.Conv1d(channels * channel_mult, channels * channel_mult,
                                    kernel_size, padding=padding),
                          nn.PReLU())
            if deeper else nn.Identity()
        )

        self.unpool2 = nn.MaxUnpool1d(2, stride=2)
        self.dec_deconv2 = nn.Conv1d(channels * channel_mult, channels,
                                     kernel_size, padding=padding)
        self.dec_act2 = nn.PReLU()
        self.dec_extra2 = (
            nn.Sequential(nn.Conv1d(channels, channels, kernel_size, padding=padding),
                          nn.PReLU())
            if deeper else nn.Identity()
        )

        self.unpool1 = nn.MaxUnpool1d(2, stride=2)
        self.dec_deconv1 = nn.Conv1d(channels, 1, kernel_size, padding=padding)
        self.dec_extra1 = (
            nn.Sequential(nn.Conv1d(1, 1, kernel_size, padding=padding))
            if deeper else nn.Identity()
        )
        self.dec_tanh = nn.Tanh()

    def encoder(self, x):
        sizes, indices = [], []

        x = self.enc_act1(self.enc_norm1(self.enc_conv1(x)))
        x = self.enc_extra1(x)
        sizes.append(x.size())
        x, idx1 = self.pool1(x)
        indices.append(idx1)

        x = self.enc_act2(self.enc_norm2(self.enc_conv2(x)))
        x = self.enc_extra2(x)
        sizes.append(x.size())
        x, idx2 = self.pool2(x)
        indices.append(idx2)

        x = self.enc_act3(self.enc_norm3(self.enc_conv3(x)))
        x = self.enc_extra3(x)
        sizes.append(x.size())
        x, idx3 = self.pool3(x)
        indices.append(idx3)

        z = self.enc_fc(x.flatten(start_dim=1))
        return z, indices, sizes

    def decoder(self, z, indices, sizes):
        idx1, idx2, idx3 = indices
        size1, size2, size3 = sizes

        x = self.dec_fc(z)
        x = x.view(z.shape[0], size3[1], -1)

        x = self.unpool3(x, idx3, output_size=size3)
        x = self.dec_act3(self.dec_deconv3(x))
        x = self.dec_extra3(x)

        x = self.unpool2(x, idx2, output_size=size2)
        x = self.dec_act2(self.dec_deconv2(x))
        x = self.dec_extra2(x)

        x = self.unpool1(x, idx1, output_size=size1)
        x = self.dec_deconv1(x)
        if self.deeper:
            x = self.dec_extra1(x)
        x = self.dec_tanh(x)
        return x

    def forward(self, x):
        z, indices, sizes = self.encoder(x)
        reconstruction = self.decoder(z, indices, sizes)
        return reconstruction, z

In [ ]:
# Probamos ambas arquitecturas con un batch pequeño antes de entrenar.
# Esto detecta errores de dimensiones sin esperar a completar una época.
with torch.no_grad():
    test_input = example_batch[:2].unsqueeze(1).to(DEVICE)

    baseline_test = AudioAutoencoder(
        input_length=INPUT_LENGTH,
        bottleneck_size=BOTTLENECK_SIZE,
        deeper=False
    ).to(DEVICE)

    improved_test = AudioAutoencoder(
        input_length=INPUT_LENGTH,
        bottleneck_size=BOTTLENECK_SIZE,
        deeper=True
    ).to(DEVICE)

    baseline_output, baseline_z = baseline_test(test_input)
    improved_output, improved_z = improved_test(test_input)

print("Entrada:", tuple(test_input.shape))
print("Salida baseline:", tuple(baseline_output.shape))
print("Bottleneck baseline:", tuple(baseline_z.shape))
print("Salida mejorada:", tuple(improved_output.shape))
print("Bottleneck mejorado:", tuple(improved_z.shape))

assert baseline_output.shape == test_input.shape
assert improved_output.shape == test_input.shape
assert baseline_z.shape[1] == BOTTLENECK_SIZE
assert improved_z.shape[1] == BOTTLENECK_SIZE

del baseline_test, improved_test, test_input, baseline_output, improved_output
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 8. Funciones de pérdida

### MSE

El error cuadrático medio calcula la media de las diferencias al cuadrado entre la señal original y la reconstruida. Es la métrica principal solicitada por el enunciado.

### Pérdida espectral multiescala

La pérdida espectral compara las magnitudes de las STFT calculadas con varias resoluciones temporales/frecuenciales. Así se penalizan diferencias en el contenido espectral a distintas escalas.

Para el segundo experimento utilizamos una combinación:

**Pérdida total = MSE + λ × pérdida espectral**

La pérdida espectral no sustituye a la evaluación MSE. Durante la validación calculamos y guardamos el MSE por separado para poder comprobar si el cambio realmente mejora el criterio del assignment.

In [ ]:
mse_criterion = nn.MSELoss()

def multi_scale_stft_loss(target, reconstruction, fft_sizes=(256, 512, 1024)):
    """Pérdida espectral relativa calculada a varias resoluciones."""
    target = target.squeeze(1)
    reconstruction = reconstruction.squeeze(1)
    losses = []

    for n_fft in fft_sizes:
        hop_length = n_fft // 4
        window = torch.hann_window(n_fft, device=target.device)

        target_stft = torch.stft(
            target, n_fft=n_fft, hop_length=hop_length,
            window=window, return_complex=True
        )
        reconstruction_stft = torch.stft(
            reconstruction, n_fft=n_fft, hop_length=hop_length,
            window=window, return_complex=True
        )

        target_magnitude = target_stft.abs()
        reconstruction_magnitude = reconstruction_stft.abs()

        # Normalizamos por la magnitud media de referencia para que las
        # diferentes resoluciones tengan escalas comparables.
        spectral_error = torch.mean(torch.abs(target_magnitude - reconstruction_magnitude))
        reference_scale = torch.mean(target_magnitude) + 1e-8
        losses.append(spectral_error / reference_scale)

    return torch.stack(losses).mean()


def count_trainable_parameters(model):
    """Cuenta únicamente los parámetros que se pueden entrenar."""
    return sum(parameter.numel() for parameter in model.parameters()
               if parameter.requires_grad)

## 9. Función de entrenamiento y validación

En cada época:
- Entrenamos con los batches de entrenamiento.
- Calculamos el MSE medio de entrenamiento.
- Evaluamos el modelo en validación sin actualizar sus parámetros.
- Guardamos el modelo que obtiene el menor MSE de validación.

El criterio de parada temprana también se basa en el MSE de validación. Esto evita seleccionar el modelo por una pérdida distinta de la que pide el enunciado.

In [ ]:
def run_epoch(model, loader, optimizer=None, use_spectral_loss=False,
              spectral_weight=0.05):
    """Ejecuta una época. Si optimizer es None, la función solo evalúa."""
    training = optimizer is not None
    model.train(training)

    total_mse = 0.0
    total_objective = 0.0
    total_examples = 0

    for batch in tqdm(loader, leave=False):
        target = batch.unsqueeze(1).to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            reconstruction, _ = model(target)
            mse = mse_criterion(reconstruction, target)

            if use_spectral_loss:
                spectral = multi_scale_stft_loss(target, reconstruction)
                objective = mse + spectral_weight * spectral
            else:
                objective = mse

            if training:
                objective.backward()
                optimizer.step()

        batch_size = target.shape[0]
        total_mse += mse.detach().item() * batch_size
        total_objective += objective.detach().item() * batch_size
        total_examples += batch_size

    return {
        "mse": total_mse / total_examples,
        "objective": total_objective / total_examples
    }


def train_model(model, train_loader, val_loader, experiment_name,
                use_spectral_loss=False):
    """Entrena, aplica early stopping y restaura el mejor MSE de validación."""
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    history = {"train_mse": [], "val_mse": [], "train_objective": [],
               "val_objective": []}
    best_val_mse = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    checkpoint_path = RESULTS_DIR / f"{experiment_name}_best.pt"

    for epoch in range(1, NUM_EPOCHS + 1):
        train_metrics = run_epoch(
            model, train_loader, optimizer,
            use_spectral_loss=use_spectral_loss
        )
        val_metrics = run_epoch(
            model, val_loader, optimizer=None,
            use_spectral_loss=use_spectral_loss
        )

        history["train_mse"].append(train_metrics["mse"])
        history["val_mse"].append(val_metrics["mse"])
        history["train_objective"].append(train_metrics["objective"])
        history["val_objective"].append(val_metrics["objective"])

        print(
            f"{experiment_name} | Epoch {epoch:02d}/{NUM_EPOCHS} | "
            f"train MSE: {train_metrics['mse']:.6f} | "
            f"val MSE: {val_metrics['mse']:.6f}"
        )

        # Guardamos el checkpoint con menor MSE de validación.
        if val_metrics["mse"] < best_val_mse:
            best_val_mse = val_metrics["mse"]
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping: no mejora durante {PATIENCE} épocas.")
            break

    # Recuperamos los pesos del checkpoint con mejor MSE de validación.
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    model.eval()

    return model, history, best_val_mse, best_epoch

## 10. Experimento A — Modelo baseline con MSE

Primero entrenamos la arquitectura de referencia, con tres bloques convolucionales y bottleneck de 512 dimensiones. La única función de pérdida utilizada para actualizar los pesos es MSE.

Este experimento sirve como punto de comparación para valorar si la arquitectura más profunda y la pérdida espectral aportan una mejora.

In [ ]:
# Reiniciamos la semilla antes de crear el modelo baseline.
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

baseline_model = AudioAutoencoder(
    input_length=INPUT_LENGTH,
    bottleneck_size=BOTTLENECK_SIZE,
    deeper=False
)

print(f"Parámetros entrenables (baseline): {count_trainable_parameters(baseline_model):,}")
baseline_model, baseline_history, baseline_best_mse, baseline_best_epoch = train_model(
    baseline_model,
    train_loader,
    val_loader,
    experiment_name="baseline_mse",
    use_spectral_loss=False
)

print(f"Mejor MSE de validación (baseline): {baseline_best_mse:.8f}")
print(f"Época seleccionada (baseline): {baseline_best_epoch}")

pd.DataFrame(baseline_history).to_csv(RESULTS_DIR / "baseline_history.csv", index_label="epoch")
pd.DataFrame([{"best_validation_mse": baseline_best_mse, "best_epoch": baseline_best_epoch,
                "trainable_parameters": count_trainable_parameters(baseline_model)}]).to_csv(RESULTS_DIR / "baseline_summary.csv", index=False)


In [ ]:
# Representamos la evolución del MSE en entrenamiento y validación.
epochs_baseline = range(1, len(baseline_history["train_mse"]) + 1)

plt.figure(figsize=(9, 4))
plt.plot(epochs_baseline, baseline_history["train_mse"], marker="o", label="Train MSE")
plt.plot(epochs_baseline, baseline_history["val_mse"], marker="o", label="Validation MSE")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("Experimento A — Baseline")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "baseline_training_validation_mse.png", dpi=200, bbox_inches="tight")
plt.show()

## 11. Experimento B — Más capas y pérdida espectral multiescala

En este experimento introducimos dos cambios permitidos por el enunciado:

1. Añadimos una convolución adicional en cada bloque de resolución del encoder y del decoder.
2. Combinamos MSE con una pérdida espectral multiescala durante el entrenamiento.

Mantenemos el mismo tamaño de segmento, las tres reducciones temporales y el bottleneck de 512 dimensiones. De esta forma, la tasa de compresión sigue siendo exactamente la misma que en el baseline.

La validación seguirá midiendo MSE, independientemente de la pérdida combinada utilizada para entrenar.

In [ ]:
# Reiniciamos la semilla para que la inicialización sea reproducible.
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

improved_model = AudioAutoencoder(
    input_length=INPUT_LENGTH,
    bottleneck_size=BOTTLENECK_SIZE,
    deeper=True
)

print(f"Parámetros entrenables (mejorado): {count_trainable_parameters(improved_model):,}")
improved_model, improved_history, improved_best_mse, improved_best_epoch = train_model(
    improved_model,
    train_loader,
    val_loader,
    experiment_name="deeper_multiscale",
    use_spectral_loss=True
)

print(f"Mejor MSE de validación (mejorado): {improved_best_mse:.8f}")
print(f"Época seleccionada (mejorado): {improved_best_epoch}")

pd.DataFrame(improved_history).to_csv(RESULTS_DIR / "improved_history.csv", index_label="epoch")
pd.DataFrame([{"best_validation_mse": improved_best_mse, "best_epoch": improved_best_epoch,
                "trainable_parameters": count_trainable_parameters(improved_model)}]).to_csv(RESULTS_DIR / "improved_summary.csv", index=False)


## 12. Comparación cuantitativa de los experimentos

La métrica principal es el menor MSE de validación alcanzado por cada modelo. Calculamos también el cambio porcentual respecto al baseline:

- Un porcentaje negativo significa que el MSE ha disminuido y, por tanto, la reconstrucción ha mejorado según el criterio del assignment.
- Un porcentaje positivo significa que el MSE ha aumentado.

No debemos asumir que el modelo B es mejor por ser más profundo o por usar una pérdida perceptual: la conclusión depende de los valores medidos.

In [ ]:
comparison = pd.DataFrame([
    {
        "Experimento": "A — Baseline (MSE)",
        "Mejor época": baseline_best_epoch,
        "Mejor validation MSE": baseline_best_mse
    },
    {
        "Experimento": "B — Más capas + STFT",
        "Mejor época": improved_best_epoch,
        "Mejor validation MSE": improved_best_mse
    }
])

relative_change = (
    (improved_best_mse - baseline_best_mse) / baseline_best_mse * 100
    if baseline_best_mse > 0 else float("nan")
)

comparison.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
display(comparison)
print(f"Cambio relativo del MSE de B respecto a A: {relative_change:+.2f}%")

if improved_best_mse < baseline_best_mse:
    print("Conclusión automática: el experimento B mejora el MSE de validación.")
elif improved_best_mse > baseline_best_mse:
    print("Conclusión automática: el experimento B no mejora el MSE de validación.")
else:
    print("Conclusión automática: ambos experimentos obtienen el mismo MSE.")

In [ ]:
# Comparamos las curvas de validación de los dos experimentos.
plt.figure(figsize=(9, 4))
plt.plot(range(1, len(baseline_history["val_mse"]) + 1),
         baseline_history["val_mse"], marker="o", label="A — Baseline")
plt.plot(range(1, len(improved_history["val_mse"]) + 1),
         improved_history["val_mse"], marker="o", label="B — Más capas + STFT")
plt.xlabel("Época")
plt.ylabel("Validation MSE")
plt.title("Comparación del MSE de validación")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "validation_mse_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

## 13. Comparación cualitativa: audio original y reconstruido

Además de la métrica, escuchamos un segmento de validación y representamos su forma de onda. Esto no sustituye al MSE, pero permite detectar artefactos evidentes y entender qué tipo de señal está reconstruyendo el modelo.

Usamos un segmento del conjunto de validación, que no se ha utilizado para actualizar los pesos.

In [ ]:
# Seleccionamos un ejemplo de validación y generamos ambas reconstrucciones.
improved_model.eval()
baseline_model.eval()

validation_audio = next(iter(val_loader))[0:1].unsqueeze(1).to(DEVICE)

with torch.no_grad():
    baseline_reconstruction, _ = baseline_model(validation_audio)
    improved_reconstruction, _ = improved_model(validation_audio)

original_np = validation_audio[0, 0].cpu().numpy()
baseline_np = baseline_reconstruction[0, 0].cpu().numpy()
improved_np = improved_reconstruction[0, 0].cpu().numpy()

print("Audio original:")
display(Audio(original_np, rate=SAMPLE_RATE))
print("Reconstrucción baseline:")
display(Audio(baseline_np, rate=SAMPLE_RATE))
print("Reconstrucción modelo mejorado:")
display(Audio(improved_np, rate=SAMPLE_RATE))

import soundfile as sf
sf.write(RESULTS_DIR / "validation_original.wav", original_np, SAMPLE_RATE)
sf.write(RESULTS_DIR / "validation_baseline_reconstruction.wav", baseline_np, SAMPLE_RATE)
sf.write(RESULTS_DIR / "validation_improved_reconstruction.wav", improved_np, SAMPLE_RATE)


In [ ]:
time_axis = np.arange(INPUT_LENGTH) / SAMPLE_RATE

plt.figure(figsize=(14, 7))

plt.subplot(3, 1, 1)
plt.plot(time_axis, original_np)
plt.title("Audio original")
plt.ylabel("Amplitud")

plt.subplot(3, 1, 2)
plt.plot(time_axis, baseline_np)
plt.title("Reconstrucción baseline")
plt.ylabel("Amplitud")

plt.subplot(3, 1, 3)
plt.plot(time_axis, improved_np)
plt.title("Reconstrucción modelo mejorado")
plt.xlabel("Tiempo (s)")
plt.ylabel("Amplitud")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "audio_reconstruction_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"MSE del ejemplo — baseline: {np.mean((original_np - baseline_np) ** 2):.8f}")
print(f"MSE del ejemplo — mejorado: {np.mean((original_np - improved_np) ** 2):.8f}")

## 14. Informe breve del assignment

Esta sección resume automáticamente los valores de los experimentos ejecutados. Es importante ejecutar las celdas anteriores antes de interpretar el informe.

### Paso 1 — Adaptación y entrenamiento en MTAT

El audio se carga desde los MP3 de MagnaTagATune. Cada ejemplo se convierte a mono, se remuestrea a 16 kHz y se recorta a un segundo (16 000 muestras). Se entrena un autoencoder convolucional en PyTorch para reconstruir el mismo segmento que recibe como entrada. No se utilizan etiquetas, porque el objetivo es la reconstrucción autosupervisada.

### Paso 2 — Tratamiento de audios largos

Se elige recortar segmentos de un segundo. En entrenamiento la posición del recorte cambia aleatoriamente, mientras que en validación se mantiene fija. Esta decisión permite procesar los clips largos con una longitud de entrada constante y conserva la configuración dimensional del modelo de referencia.

### Paso 3 — Mejora de la reconstrucción

Se compara el modelo baseline con una segunda arquitectura que incorpora convoluciones adicionales y una pérdida espectral multiescala combinada con MSE. En ambos casos el bottleneck mantiene 512 dimensiones y la tasa de compresión se mantiene en 31,25× según la definición del notebook de clase.

**Resultados medidos:** la tabla y el cambio porcentual que aparecen a continuación se calculan con los valores reales obtenidos durante la ejecución. La interpretación debe basarse en ellos, no en la complejidad de la arquitectura.

In [ ]:
report_text = f"""
### Resultados experimentales

| Modelo | Época seleccionada | Mejor MSE de validación |
|---|---:|---:|
| Baseline (MSE) | {baseline_best_epoch} | {baseline_best_mse:.8f} |
| Más capas + STFT | {improved_best_epoch} | {improved_best_mse:.8f} |

El cambio relativo del MSE del segundo modelo respecto al baseline es **{relative_change:+.2f}%**.

"""

if improved_best_mse < baseline_best_mse:
    interpretation = (
        "En esta ejecución, el modelo con más capas y pérdida espectral obtiene un "
        "MSE de validación menor. Por tanto, mejora el criterio cuantitativo solicitado."
    )
elif improved_best_mse > baseline_best_mse:
    interpretation = (
        "En esta ejecución, el modelo con más capas y pérdida espectral obtiene un "
        "MSE de validación mayor. Por tanto, las modificaciones probadas no mejoran "
        "el criterio cuantitativo solicitado, aunque puedan producir diferencias "
        "perceptuales que habría que estudiar por separado."
    )
else:
    interpretation = (
        "En esta ejecución, ambos modelos obtienen el mismo MSE de validación."
    )

report_text += f"### Interpretación\\n\\n{interpretation}\\n\\n"
report_text += (
    "La comparación es válida porque ambos modelos utilizan los mismos segmentos, "
    "la misma partición de validación, la misma longitud de entrada y el mismo "
    "bottleneck de 512 valores. Los resultados corresponden a esta ejecución y "
    "pueden variar ligeramente con la inicialización, el hardware y las versiones "
    "de las librerías."
)

display(Markdown(report_text))
(RESULTS_DIR / "experimental_report.md").write_text(report_text, encoding="utf-8")

## 15. Comprobaciones finales y reproducibilidad

Antes de entregar:

- Ejecuta el notebook desde el principio en un entorno limpio de Colab.
- Comprueba que se han descargado y encontrado los MP3 de MTAT.
- Verifica que se han completado los dos entrenamientos y que se muestra la tabla de comparación.
- Conserva las salidas de las celdas con las métricas y las gráficas.
- Si Colab interrumpe la sesión, vuelve a ejecutar la descarga (wget reanuda las partes que ya existen) y continúa desde el entrenamiento.

### Limitaciones

- Se utiliza una partición aleatoria a nivel de clip, no una partición por artista o canción original.
- Los segmentos de validación se extraen siempre del centro de cada clip; los segmentos de entrenamiento se muestrean aleatoriamente.
- El MSE es la métrica principal del assignment, pero no siempre coincide con la calidad perceptual del audio.
- El entrenamiento completo puede tardar bastante debido al tamaño de MTAT y a la decodificación de MP3.

### Referencias

- Material de clase: *Self-supervised learning and autoencoders*, Class 6 (2026), diapositivas 37–38.
- Notebook de referencia: [06_01_autoencoders.ipynb](https://github.com/MTG/machine_learning_for_sound_and_music_25/blob/main/notebooks/06_01_autoencoders.ipynb).
- Dataset: [The MagnaTagATune Dataset — MIRG](https://mirg.city.ac.uk/codeapps/the-magnatagatune-dataset).
- Pérdida espectral multiescala: [Parallel WaveGAN (paper)](https://arxiv.org/abs/1910.11480).